<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 21 · 服务与模型出问题时发生什么

一次上下文请求失败，可能是没有材料，也可能是服务根本不可用。这两种情况不能混淆。本篇记录真实 HTTP 与模型处理的指标和追踪，故意让模型连接失败，再恢复配置；最后停止 Server，观察 Agent 工具如何报告不可用。

需要真实 Generation 和对话模型。故障使用本机未监听的端口，不向陌生地址发送请求。追踪保存在本次进程中的 OpenTelemetry exporter，不连接外部观测平台。

路线：指标和 trace → 真实连接失败 → 保留未处理材料 → 恢复真实模型 → Agent 在服务故障时继续回答。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))
if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("21", features=("generation",))
client = lab.client
assert client is not None
scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 21", summary="本次教学实验的独立材料", idempotency_key=f"{lab.run_id}:main"
    )
)
scope_id = scope.scope_id

## 先让请求留下可关联的记录

这里使用标准 OpenTelemetry SDK 的本地 exporter 收集实际 span。它只是本次实验的观测目的地，不会替换业务操作或模型实现。

In [ ]:
import httpx
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

from powercontext.http import CreateSourceRequest, PrepareContextRequest
from powercontext.server.settings import MetricsConfig
from powercontext.server.tracing import ServerTracing

exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))
lab.app_options["tracing"] = ServerTracing(provider, instrumented=True)
lab.settings_overrides["metrics"] = MetricsConfig(enabled=True)
await lab.restart()
client = lab.client
source = await client.create_source(
    scope_id, CreateSourceRequest(content="amount: 本项目长期约定是金额存储为整数分，拒绝负数和超过两位小数的输入。")
)
empty = await client.prepare_context(PrepareContextRequest(scope_id=scope_id, query="amount"))
assert empty.status == "empty"
async with httpx.AsyncClient(base_url=lab.base_url) as http:
    metrics = await http.get("/metrics")
    metrics.raise_for_status()
assert "powercontext_server_transport_requests_total" in metrics.text
spans = exporter.get_finished_spans()
assert spans
show({"实际 span 数": len(spans), "样例 span": [span.name for span in spans[:8]]})

## 让 Generation 连接失败，检查 Source 是否仍待处理

保存正常配置，再指向本机未监听端口。模型连接失败必须作为错误显现，不能返回一份伪造的成功提取。服务重启会沿用原实验数据库。

In [ ]:
import socket

from powercontext.client import ClientError
from powercontext.http import FlushMemoryRequest, ListMemoryEntriesRequest

real_inference = lab.inference
listener = socket.socket()
listener.bind(("127.0.0.1", 0))
closed_port = listener.getsockname()[1]
listener.close()
lab.inference = real_inference.model_copy(
    update={"generation_base_url": f"http://127.0.0.1:{closed_port}/v1", "generation_timeout_seconds": 2}
)
await lab.restart()
client = lab.client
try:
    await client.flush_memory(FlushMemoryRequest(scope_id=scope_id))
except ClientError as error:
    failure_kind = type(error).__name__
else:
    raise AssertionError("模型不可连接时不能报告成功")
assert (await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))).entries == []
retained = await client.get_source(scope_id, "content", source.source_id)
assert retained.source_id == source.source_id
show({"实际失败类型": failure_kind, "Source 仍可读": True, "没有伪造 Memory": True})

## 恢复真实模型，处理同一份材料

不重新写入 Source，也不改写模型结果。恢复后手动 flush 这份仍待处理的材料，检查真实来源；之后无新材料的 flush 不应重复产生条目。

In [ ]:
import json

lab.inference = real_inference
await lab.restart()
client = lab.client
recovered = await client.flush_memory(FlushMemoryRequest(scope_id=scope_id))
entries = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
assert entries.entries
assert any(any(ref.source_id == source.source_id for ref in entry.source_refs) for entry in entries.entries)
again = await client.flush_memory(FlushMemoryRequest(scope_id=scope_id))
after = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
assert [item.citation for item in after.entries] == [item.citation for item in entries.entries]
show({"恢复后 Memory 条数": len(entries.entries), "重复处理未增加条目": True})
async with httpx.AsyncClient(base_url=lab.base_url) as http:
    metrics = await http.get("/metrics")
(lab.directory / "metrics.txt").write_text(metrics.text)
span_rows = [
    {"name": span.name, "trace_id": format(span.context.trace_id, "032x"), "status": span.status.status_code.name}
    for span in exporter.get_finished_spans()
]
(lab.directory / "spans.json").write_text(json.dumps(span_rows, indent=2))
table(span_rows[-12:])

## Server 不可用时，工具给出错误而不是空记忆

停止 Server，但保留真实对话模型。Agent 仍能收到工具返回的不可用信息，然后向用户说明缺少项目依据。我们检查原始工具消息，避免只根据模型的最终措辞下结论。

In [ ]:
from _tutorial import chat_settings
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from powercontext_langgraph import PowerContextScope, powercontext_tools

old_url = lab.base_url
await lab.close()
agent = create_agent(
    ChatOpenAI(**chat_settings()),
    tools=powercontext_tools(),
    context_schema=PowerContextScope,
    system_prompt="Use the search tool once for project facts. If unavailable, explain that project memory cannot currently be read; do not invent facts or keep retrying.",
)
result = await agent.ainvoke(
    {"messages": [("user", "请通过 powercontext_search 查询 amount 项目约定。如果服务不可用，告诉我实际限制。")]},
    context=PowerContextScope(scope_id=scope_id, base_url=old_url, timeout=2),
)
tool_messages = [message for message in result["messages"] if message.type == "tool"]
assert tool_messages and any("PowerContext unavailable" in message.text for message in tool_messages)
assert all("no matching PowerContext memory" not in message.text for message in tool_messages)
print(result["messages"][-1].text)
provider.shutdown()

## 练习与验收

比较空 Scope 的 empty 与服务停止后的 unavailable。前者有一次成功查询作为依据，后者没有完成查询。若接入外部 OTLP 平台，可沿用相同追踪接口；不要把配置凭证或原始私人材料写进输出。

接下来阅读 [22_complete_team_workflow.ipynb](22_complete_team_workflow.ipynb)。

最后关闭服务。实验文件保留在本次 `.powercontext/` 目录，便于复查。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")